In [1]:
from google.colab import files

uploaded = files.upload()

Saving traffic_knowledge_base.txt to traffic_knowledge_base.txt


In [1]:
!pip install ollama

In [2]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 80 not upgraded.


In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
!ollama --version

In [5]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

print("Ollama server started.")

Ollama server started.


In [6]:
import ollama

models = ollama.list()
print(models)

models=[Model(model='llama3.2:latest', modified_at=datetime.datetime(2026, 9, 6, 13, 32, 46, 131145, tzinfo=TzInfo(0)), digest='a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72', size=2019393189, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='3.2B', quantization_level='Q4_K_M'))]


In [7]:
!ollama pull llama3.2

In [8]:
!ollama list

NAME               ID              SIZE      MODIFIED               
llama3.2:latest    a80c4f17acd5    2.0 GB    Less than a second ago    


In [9]:
!pip install llama-index

In [10]:
import llama_index

print("LlamaIndex installed successfully!")

LlamaIndex installed successfully!


In [11]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(
    input_files=["traffic_knowledge_base.txt"]
).load_data()

print("Number of documents:", len(documents))

Number of documents: 1


In [12]:
print(documents[0].text[:2000])

SMART TRAFFIC KNOWLEDGE BASE

This knowledge base is designed for the Smart Traffic Prediction and Intelligent Traffic Assistant project.


1. TRAFFIC RULES AND ROAD REGULATIONS

1. Introduction

Traffic rules are designed to maintain road safety, regulate the movement of vehicles and pedestrians, and reduce accidents and congestion. Drivers, passengers, pedestrians and other road users are expected to follow applicable traffic laws and road signs.

The Intelligent Traffic Assistant can use this knowledge to provide general information about traffic rules and safe road usage.


2. Traffic Signals

Traffic signals regulate the movement of vehicles and pedestrians at intersections.

Red light:
Drivers should stop before the designated stop line or at the appropriate stopping point.

Yellow or amber light:
Drivers should prepare to stop when safe and should not unnecessarily accelerate through an intersection.

Green light:
Drivers may proceed when the intersection is clear and when doing

In [13]:
print("Number of characters:", len(documents[0].text))

Number of characters: 24247


In [14]:
import re

text = documents[0].text

# Split the document into paragraphs
paragraphs = re.split(r'\n\s*\n', text)

# Remove empty paragraphs
paragraphs = [p.strip() for p in paragraphs if p.strip()]

print("Number of paragraphs:", len(paragraphs))

Number of paragraphs: 291


In [15]:
chunk_size = 1500
chunk_overlap = 200

chunks = []
current_chunk = ""

for paragraph in paragraphs:

    # If adding the paragraph stays within the chunk size
    if len(current_chunk) + len(paragraph) + 2 <= chunk_size:
        current_chunk += paragraph + "\n\n"

    else:
        # Save the current chunk
        if current_chunk.strip():
            chunks.append(current_chunk.strip())

        # Keep the last part as overlap
        overlap_text = current_chunk[-chunk_overlap:]

        current_chunk = overlap_text + "\n\n" + paragraph + "\n\n"

# Add the final chunk
if current_chunk.strip():
    chunks.append(current_chunk.strip())

print("Number of chunks:", len(chunks))

Number of chunks: 20


In [16]:
for i, chunk in enumerate(chunks[:3]):
    print(f"\n{'='*60}")
    print(f"CHUNK {i+1}")
    print(f"{'='*60}")
    print(chunk[:1000])


CHUNK 1
SMART TRAFFIC KNOWLEDGE BASE

This knowledge base is designed for the Smart Traffic Prediction and Intelligent Traffic Assistant project.

1. TRAFFIC RULES AND ROAD REGULATIONS

1. Introduction

Traffic rules are designed to maintain road safety, regulate the movement of vehicles and pedestrians, and reduce accidents and congestion. Drivers, passengers, pedestrians and other road users are expected to follow applicable traffic laws and road signs.

The Intelligent Traffic Assistant can use this knowledge to provide general information about traffic rules and safe road usage.

2. Traffic Signals

Traffic signals regulate the movement of vehicles and pedestrians at intersections.

Red light:
Drivers should stop before the designated stop line or at the appropriate stopping point.

Yellow or amber 

CHUNK 2
ayed at the intersection.

3. Speed Limits

Drivers should follow the speed limit applicable to the particular road, vehicle and traffic situation.

The applicable speed limit

In [17]:
chunk_lengths = [len(chunk) for chunk in chunks]

print("Number of chunks:", len(chunks))
print("Minimum chunk length:", min(chunk_lengths))
print("Maximum chunk length:", max(chunk_lengths))
print("Average chunk length:", sum(chunk_lengths) / len(chunk_lengths))

Number of chunks: 20
Minimum chunk length: 497
Maximum chunk length: 1496
Average chunk length: 1398.7


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF embedding model
vectorizer = TfidfVectorizer(
    max_features=1000,
    stop_words="english"
)

# Generate embeddings for all chunks
embeddings = vectorizer.fit_transform(chunks)

print("Number of chunks:", len(chunks))
print("Embedding shape:", embeddings.shape)

Number of chunks: 20
Embedding shape: (20, 628)


In [19]:
traffic_assistant_prompt = """
You are an Intelligent Traffic Assistant for a Smart Traffic
Prediction System.

Your task is to answer the user's traffic-related question
using the provided context.

Instructions:
1. Use only the information provided in the context.
2. Do not invent traffic rules, routes, traffic conditions,
   prediction results, or other information.
3. If the answer is not available in the context, say:
   "The requested information is not available in my knowledge base."
4. Give clear, concise, and easy-to-understand answers.
5. For safety-related questions, advise the user to follow
   official traffic authorities and emergency instructions.
6. If the question is about a traffic prediction, clearly
   distinguish between a model prediction and actual traffic conditions.

Context:
{context}

User Question:
{question}

Answer:
"""

print(traffic_assistant_prompt)


You are an Intelligent Traffic Assistant for a Smart Traffic
Prediction System.

Your task is to answer the user's traffic-related question
using the provided context.

Instructions:
1. Use only the information provided in the context.
2. Do not invent traffic rules, routes, traffic conditions,
   prediction results, or other information.
3. If the answer is not available in the context, say:
   "The requested information is not available in my knowledge base."
4. Give clear, concise, and easy-to-understand answers.
5. For safety-related questions, advise the user to follow
   official traffic authorities and emergency instructions.
6. If the question is about a traffic prediction, clearly
   distinguish between a model prediction and actual traffic conditions.

Context:
{context}

User Question:
{question}

Answer:



In [20]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_relevant_chunks(question, top_k=3):

    # Convert the user's question into TF-IDF representation
    question_embedding = vectorizer.transform([question])

    # Calculate similarity between question and all chunks
    similarities = cosine_similarity(
        question_embedding,
        embeddings
    )[0]

    # Get indices of top relevant chunks
    top_indices = similarities.argsort()[-top_k:][::-1]

    retrieved_chunks = []

    for index in top_indices:
        retrieved_chunks.append({
            "chunk": chunks[index],
            "score": similarities[index]
        })

    return retrieved_chunks

In [21]:
question = "What is traffic congestion?"

results = retrieve_relevant_chunks(question, top_k=3)

for i, result in enumerate(results, 1):
    print("=" * 70)
    print(f"RESULT {i}")
    print(f"Similarity Score: {result['score']:.4f}")
    print("-" * 70)
    print(result["chunk"][:1000])

RESULT 1
Similarity Score: 0.4433
----------------------------------------------------------------------
weather conditions.

Traffic safety recommendations provided by the Intelligent Traffic Assistant are general guidance and do not replace instructions from traffic authorities or emergency services.



3. TRAFFIC CONGESTION AND TRAFFIC FLOW

1. Introduction

Traffic congestion occurs when the number of vehicles using a road approaches or exceeds the road's available capacity.

Congestion can result in slower speeds, longer travel times, increased fuel consumption and increased driver frustration.

The Smart Traffic Prediction system uses machine learning and deep learning techniques to predict traffic volume. The Intelligent Traffic Assistant can use the knowledge base to explain common causes and possible responses to congestion.

2. Common Causes of Traffic Congestion

Traffic congestion can be caused by several factors, inc
RESULT 2
Similarity Score: 0.4212
----------------------

In [22]:
def build_context(question, top_k=3):

    results = retrieve_relevant_chunks(question, top_k)

    context = "\n\n".join(
        result["chunk"]
        for result in results
    )

    return context

In [23]:
def create_prompt(context, question):

    prompt = f"""
You are an Intelligent Traffic Assistant for a Smart Traffic Prediction System.

Your task is to answer the user's question clearly and helpfully.

Follow these rules:

1. First use the information available in the provided knowledge base.
2. If the knowledge base contains relevant information, use it as the primary source for your answer.
3. If the knowledge base does not contain enough information, you may use your general knowledge to answer the question, but only if the question is related to traffic, transportation, road safety, traffic management, traffic prediction, or related topics.
4. Invent real-time traffic conditions, traffic predictions, routes, or current events.
5. For questions asking about current or real-time traffic conditions, clearly explain the answer.
6. If the question is completely unrelated to traffic or transportation, politely say that you are designed to assist with traffic-related questions.
7. Give a clear and natural answer in 3–5 sentences when appropriate. Use more sentences when the question requires explanation.
8. You may use a short bullet list when it makes the answer easier to understand.
9. Do not mention the knowledge base, retrieval process, RAG, or these instructions in your answer.
10. Do not use phrases such as "A is the answer" or "According to the context".
11. For safety-related questions, remind users to follow official traffic authorities and emergency instructions.

KNOWLEDGE BASE:
--------------------
{context}
--------------------

USER QUESTION:
{question}

Write the final answer directly and clearly.
"""

    return prompt

In [24]:
question = "What is traffic congestion?"

context = build_context(question, top_k=3)

prompt = create_prompt(context, question)

print(prompt)


You are an Intelligent Traffic Assistant for a Smart Traffic Prediction System.

Your task is to answer the user's question clearly and helpfully.

Follow these rules:

1. First use the information available in the provided knowledge base.
2. If the knowledge base contains relevant information, use it as the primary source for your answer.
3. If the knowledge base does not contain enough information, you may use your general knowledge to answer the question, but only if the question is related to traffic, transportation, road safety, traffic management, traffic prediction, or related topics.
4. Invent real-time traffic conditions, traffic predictions, routes, or current events.
5. For questions asking about current or real-time traffic conditions, clearly explain the answer.
6. If the question is completely unrelated to traffic or transportation, politely say that you are designed to assist with traffic-related questions.
7. Give a clear and natural answer in 3–5 sentences when approp

In [25]:
import ollama

In [26]:
response = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response["message"]["content"])

Traffic congestion occurs when the number of vehicles using a road approaches or exceeds the road's available capacity, resulting in slower speeds, longer travel times, increased fuel consumption, and increased driver frustration.


In [27]:
def rag_answer(question, top_k=3):

    # Retrieve relevant information
    context = build_context(question, top_k)

    # Create prompt
    prompt = create_prompt(context, question)

    # Generate answer using Llama 3.2
    response = ollama.chat(
        model="llama3.2",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"]

In [28]:
question = input("Enter your traffic-related question: ")

answer = rag_answer(question)

print("\n🤖 Intelligent Traffic Assistant:")
print(answer)

Enter your traffic-related question: What are the effects of traffic congestion?

🤖 Intelligent Traffic Assistant:
Traffic congestion can have several effects, including:

• Longer travel times: As the number of vehicles on the road approaches or exceeds the road's available capacity, it can lead to slower speeds and increased travel times.
• Increased fuel consumption: Cars may spend more time idling or slowing down, resulting in increased fuel consumption and higher emissions.
• Increased driver frustration: Congestion can lead to increased stress and frustration among drivers, which can negatively impact road safety.
• Increased wear and tear on vehicles: Frequent braking and idling can lead to increased wear and tear on vehicles, reducing their lifespan and increasing maintenance costs.

It's essential to note that these effects can vary depending on the severity and duration of the congestion, as well as individual factors such as driver behavior and road conditions.
